In [9]:
"""
Health&You - Excel Database Generator
-------------------------------------------------------------
"""

import random
import hashlib
from datetime import datetime, timedelta, date
from faker import Faker
import pandas as pd


SEED = 42
random.seed(SEED)

fake_fr = Faker("fr_FR")
fake_be = Faker("fr_BE")
Faker.seed(SEED)
fake_fr.seed_instance(SEED)
fake_be.seed_instance(SEED)

OUTPUT_XLSX = "/tmp/healthyou_db.xlsx"

N_USERS = 10_000
N_HEALTH_USERS = 1_000
HEALTH_DAYS = 90

N_CHAT = 5_000
N_LOGS = 50_000

N_PASSWORD_RESETS = 7_499
N_SUPPORT_CALLS = 10_234
N_SOCIAL_MENTIONS = 6_169

N_SUSP_FOUND = 2_207

N_SUSP_TOTAL = 2_456

N_WEAK_PASSWORD_USERS = 2_988

FIXED_NOW = datetime(2025, 12, 12, 13, 4, 1)

# HELPERS

def md5_hash(s: str) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def iso_ts(dt: datetime) -> str:
    return dt.isoformat(timespec="microseconds")

def random_dt_between(start: datetime, end: datetime) -> datetime:
    if end <= start:
        return start
    delta = end - start
    seconds = int(delta.total_seconds())
    return start + timedelta(seconds=random.randint(0, seconds), microseconds=random.randint(0, 999999))

def strong_password(length: int = 10) -> str:
    alphabet = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789!@#$%^&*()_+-=<>?"
    return "".join(random.choice(alphabet) for _ in range(length))

def format_address(fake: Faker) -> str:
    return f"{fake.street_address()}, {fake.postcode()} {fake.city()}"

def pick_country_list(n: int, frac_fr: float = 0.706):
    n_fr = int(round(n * frac_fr))
    n_be = n - n_fr
    arr = (["France"] * n_fr) + (["Belgium"] * n_be)
    random.shuffle(arr)
    return arr

# GENERATORS

def generate_users():
    countries = pick_country_list(N_USERS, frac_fr=0.706)

    n_premium = int(round(N_USERS * 0.1492))
    premium_flags = ([True] * n_premium) + ([False] * (N_USERS - n_premium))
    random.shuffle(premium_flags)

    weak_password_pool = (
        ["123456"] * 618
        + ["password"] * 612
        + ["azerty123"] * 597
        + ["admin123"] * 585
        + ["motdepasse"] * 576
    )
    assert len(weak_password_pool) == N_WEAK_PASSWORD_USERS
    random.shuffle(weak_password_pool)

    weak_idx_set = set(random.sample(range(N_USERS), N_WEAK_PASSWORD_USERS))

    users = []
    reg_start = datetime(2023, 11, 26, 0, 0, 0)
    reg_end = datetime(2025, 11, 24, 23, 59, 59)

    login_start = datetime(2025, 8, 28, 14, 0, 0)
    login_end = datetime(2025, 11, 27, 14, 5, 0)

    weak_pw_cursor = 0

    for i in range(N_USERS):
        user_id = i + 1
        country = countries[i]
        fake = fake_fr if country == "France" else fake_be

        first_name = fake.first_name()
        last_name = fake.last_name()
        email = fake.email()

        dob = fake.date_of_birth(minimum_age=18, maximum_age=75).strftime("%Y-%m-%d")

        phone = fake.phone_number()
        address = format_address(fake)

        registration_date = random_dt_between(reg_start, reg_end).date().isoformat()

        last_login_dt = random_dt_between(login_start, login_end)
        last_login = iso_ts(last_login_dt)

        if i in weak_idx_set:
            pw_plain = weak_password_pool[weak_pw_cursor]
            weak_pw_cursor += 1
        else:
            pw_plain = strong_password(10)

        pw_hash = md5_hash(pw_plain)

        users.append({
            "user_id": user_id,
            "first_name": first_name,
            "last_name": last_name,
            "email": email,
            "password_hash": pw_hash,
            "password_plain": pw_plain,
            "date_of_birth": dob,
            "phone": phone,
            "address": address,
            "registration_date": registration_date,
            "country": country,
            "is_premium": bool(premium_flags[i]),
            "last_login": last_login
        })

    return pd.DataFrame(users)

def generate_health_data(df_users: pd.DataFrame):
    health_user_ids = random.sample(df_users["user_id"].tolist(), N_HEALTH_USERS)
    health_rows = []

    user_last_login = dict(zip(df_users["user_id"], df_users["last_login"]))

    for uid in health_user_ids:
        end_dt = datetime.fromisoformat(user_last_login[uid])
        end_day = end_dt.date()

        for d in range(HEALTH_DAYS):
            day = end_day - timedelta(days=d)
            ts = datetime.combine(day, end_dt.time()) + timedelta(
                minutes=random.randint(-120, 120),
                seconds=random.randint(0, 59),
                microseconds=random.randint(0, 999999)
            )

            hr_min = random.randint(45, 70)
            hr_max = random.randint(90, 165)
            hr_avg = max(hr_min + 5, min(hr_max - 5, random.randint(55, 95)))

            steps = random.randint(500, 18000)
            distance = round(steps * random.uniform(0.0006, 0.00085), 2)  # rough km/step conversion
            calories = int(steps * random.uniform(0.03, 0.06)) + random.randint(0, 250)

            sleep_hours = round(random.uniform(4.0, 9.5), 1)
            sleep_quality = random.randint(40, 100)

            water = random.randint(200, 2500)
            weight = round(random.uniform(48.0, 125.0), 1)

            health_rows.append({
                "user_id": uid,
                "date": day.isoformat(),
                "timestamp": iso_ts(ts),
                "heart_rate_avg": hr_avg,
                "heart_rate_max": hr_max,
                "heart_rate_min": hr_min,
                "steps_count": steps,
                "distance_km": distance,
                "calories_burned": calories,
                "sleep_duration_hours": sleep_hours,
                "sleep_quality_score": sleep_quality,
                "water_intake_ml": water,
                "weight_kg": weight
            })

    df_health = pd.DataFrame(health_rows)
    df_health.sort_values(["user_id", "date"], ascending=[True, False], inplace=True)
    df_health.reset_index(drop=True, inplace=True)
    return df_health

def generate_chatbot(df_users: pd.DataFrame):
    templates = [
        ("Problèmes digestifs", "Il est important de faire des analyses sanguines pour vérifier."),
        ("Mal de tête fréquent", "Je vous recommande de consulter un médecin si les symptômes persistent."),
        ("Anxiété et stress", "Vos données semblent dans la norme, mais consultez un professionnel."),
        ("J'ai des palpitations", "Surveillez votre fréquence cardiaque et reposez-vous aujourd’hui."),
        ("Je dors mal", "Essayez une routine régulière et évitez les écrans avant de dormir."),
        ("Douleur au genou", "Si la douleur persiste, consultez un professionnel de santé."),
        ("Je suis très fatigué(e)", "Un bilan sanguin peut aider à identifier une cause possible."),
        ("Essoufflement", "Allez-y progressivement et consultez si cela s’aggrave."),
        ("Vertiges", "Hydratez-vous et levez-vous lentement. Consultez si cela continue."),
        ("Crampes fréquentes", "Le magnésium et l’hydratation peuvent aider.")
    ]
    sentiments = ["positive", "neutral", "negative"]

    rows = []
    ts_start = datetime(2025, 8, 28, 14, 0, 0)
    ts_end = datetime(2025, 11, 26, 14, 5, 0)

    user_map = df_users.set_index("user_id")["email"].to_dict()

    for cid in range(1, N_CHAT + 1):
        uid = random.randint(1, N_USERS)
        uemail = user_map[uid]
        ts = random_dt_between(ts_start, ts_end)

        umsg, bmsg = random.choice(templates)
        conv_len = random.randint(1, 10)

        rows.append({
            "conversation_id": cid,
            "user_id": uid,
            "user_email": uemail,
            "timestamp": iso_ts(ts),
            "user_message": umsg,
            "bot_response": bmsg,
            "sentiment": random.choice(sentiments),
            "conversation_length": conv_len
        })

    return pd.DataFrame(rows)

def generate_access_logs():
    endpoints = ["/_search", "/health", "/login", "/api/v1/users", "/api/v1/metrics"]
    methods = ["GET", "POST"]
    user_agents = [
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)",
        "python-requests/2.28.1",
        "curl/7.68.0",
        "Mozilla/5.0 (compatible; Nmap Scripting Engine)"
    ]

    ts_start = datetime(2025, 8, 28, 15, 3, 0)
    ts_end = datetime(2025, 11, 27, 14, 2, 1)

    logs = []
    flags = ([True] * N_SUSP_TOTAL) + ([False] * (N_LOGS - N_SUSP_TOTAL))
    random.shuffle(flags)

    susp_positions = [i for i, f in enumerate(flags) if f]
    found_positions = set(random.sample(susp_positions, N_SUSP_FOUND))

    for i in range(N_LOGS):
        log_id = i + 1
        is_susp = flags[i]

        ts = random_dt_between(ts_start, ts_end)

        if is_susp:
            endpoint = "/_search"
            qp = "*"
            is_auth = False
            method = "GET"
            status = 200
            if i in found_positions:
                resp_size = random.randint(50_000, 500_000)
            else:
                resp_size = random.randint(1_000, 49_999)
            ip = f"{random.randint(1, 255)}.{random.randint(0, 255)}.{random.randint(0, 255)}.{random.randint(0, 255)}"
            ua = random.choice(["python-requests/2.28.1", "curl/7.68.0", "Mozilla/5.0 (compatible; Nmap Scripting Engine)"])
        else:
            endpoint = random.choice(endpoints)
            qp = random.choice(["", "page=1", "limit=50", "q=steps", "q=sleep"])
            is_auth = random.random() < 0.85
            method = random.choice(methods)
            status = random.choice([200, 200, 200, 201, 204, 400, 401, 403, 404, 500])
            resp_size = random.randint(500, 200_000)
            ip = f"82.{random.randint(0, 255)}.{random.randint(0, 255)}.{random.randint(0, 255)}"
            ua = random.choice(user_agents)

        logs.append({
            "log_id": log_id,
            "timestamp": iso_ts(ts),
            "source_ip": ip,
            "user_agent": ua,
            "request_method": method,
            "endpoint": endpoint,
            "status_code": status,
            "response_size_bytes": resp_size,
            "query_parameters": qp,
            "is_authenticated": bool(is_auth),
            "is_suspicious": bool(is_susp)
        })

    df_logs = pd.DataFrame(logs)
    df_logs.sort_values("timestamp", inplace=True)
    df_logs.reset_index(drop=True, inplace=True)

    return df_logs

def derive_suspicious_found(df_logs: pd.DataFrame):
    df = df_logs[(df_logs["is_suspicious"] == True) & (df_logs["response_size_bytes"] >= 50_000)].copy()
    if len(df) > N_SUSP_FOUND:
        df = df.sample(N_SUSP_FOUND, random_state=SEED).copy()
    elif len(df) < N_SUSP_FOUND:
        extra = df_logs[(df_logs["is_suspicious"] == True) & (~df_logs.index.isin(df.index))].sample(
            N_SUSP_FOUND - len(df), random_state=SEED
        )
        df = pd.concat([df, extra], ignore_index=True)
    df.reset_index(drop=True, inplace=True)
    return df

def generate_password_resets(df_users: pd.DataFrame):
    n_pending = 376
    n_completed = N_PASSWORD_RESETS - n_pending

    statuses = (["completed"] * n_completed) + (["pending"] * n_pending)
    random.shuffle(statuses)

    reset_types = ["email_link", "sms_code", "support_assisted"]
    ts_start = datetime(2025, 8, 28, 15, 3, 0)
    ts_end = datetime(2025, 11, 26, 14, 5, 0)

    rows = []
    for rid in range(1, N_PASSWORD_RESETS + 1):
        uid = random.randint(1, N_USERS)
        status = statuses[rid - 1]

        if status == "pending":
            ts_val = None
        else:
            ts_val = iso_ts(random_dt_between(ts_start, ts_end))

        rows.append({
            "reset_id": rid,
            "user_id": uid,
            "timestamp": ts_val,
            "reset_type": random.choice(reset_types),
            "status": status,
            "mfa_enabled_after": bool(random.random() < 0.55)
        })

    return pd.DataFrame(rows)

def generate_crisis_notifications(df_susp_found: pd.DataFrame, df_password_resets: pd.DataFrame):
    rows = []
    nid = 1

    for _, r in df_susp_found.iterrows():
        rows.append({
            "notification_id": nid,
            "timestamp": iso_ts(datetime.fromisoformat(r["timestamp"]) + timedelta(seconds=1)),
            "type": "security_alert",
            "recipient": "security_team@healthandyou.example",
            "status": "sent",
            "priority": "high",
            "triggered_by": f"suspicious_access_{int(r['log_id'])}"
        })
        nid += 1

    for _, r in df_password_resets.iterrows():
        if r["timestamp"] is None or (isinstance(r["timestamp"], float) and pd.isna(r["timestamp"])):
            ts = FIXED_NOW - timedelta(days=random.randint(0, 20), minutes=random.randint(0, 180))
        else:
            ts = datetime.fromisoformat(r["timestamp"]) + timedelta(minutes=1)

        rows.append({
            "notification_id": nid,
            "timestamp": iso_ts(ts),
            "type": "user_notification",
            "recipient": f"user_{int(r['user_id'])}@email",
            "status": "sent" if r["status"] == "completed" else "queued",
            "priority": "high",
            "triggered_by": f"password_reset_{int(r['reset_id'])}"
        })
        nid += 1

    inv_ts_start = datetime(2025, 11, 28, 10, 0, 0)
    inv_ts_end = datetime(2025, 12, 1, 18, 0, 0)
    for i in range(12):
        ts = random_dt_between(inv_ts_start, inv_ts_end)
        rows.append({
            "notification_id": nid,
            "timestamp": iso_ts(ts),
            "type": "investor_alert",
            "recipient": "investors@healthandyou.example",
            "status": "sent",
            "priority": "critical",
            "triggered_by": "board_escalation"
        })
        nid += 1

    rows.append({
        "notification_id": nid,
        "timestamp": iso_ts(datetime(2025, 11, 30, 16, 4, 1, 213731)),
        "type": "cnil_notification",
        "recipient": "cniL@notification.example",
        "status": "sent",
        "priority": "critical",
        "triggered_by": "gdpr_article_33"
    })
    nid += 1

    df = pd.DataFrame(rows)
    df = df.iloc[:9719].copy()
    df.reset_index(drop=True, inplace=True)
    return df

def generate_support_calls():
    categories = ["security", "account", "billing", "technical", "privacy"]
    ts_start = datetime(2025, 11, 28, 0, 4, 0)
    ts_end = datetime(2025, 12, 12, 13, 2, 1)

    rows = []
    for cid in range(1, N_SUPPORT_CALLS + 1):
        ts = random_dt_between(ts_start, ts_end)
        dur = random.randint(2, 45)
        resolved = random.random() < 0.85
        rating = random.randint(1, 5) if resolved else random.randint(1, 4)
        rows.append({
            "call_id": cid,
            "timestamp": iso_ts(ts),
            "category": random.choice(categories),
            "duration_minutes": dur,
            "resolved": bool(resolved),
            "satisfaction_rating": rating
        })
    return pd.DataFrame(rows)

def generate_social_mentions():
    platforms = ["X", "Facebook", "Instagram", "Reddit", "TikTok", "YouTube"]
    sentiments = ["positive", "neutral", "negative"]
    ts_start = datetime(2025, 11, 27, 16, 4, 0)
    ts_end = datetime(2025, 12, 12, 13, 4, 0)

    rows = []
    for mid in range(1, N_SOCIAL_MENTIONS + 1):
        ts = random_dt_between(ts_start, ts_end)
        reach = random.randint(100, 50000)
        engagement = int(reach * random.uniform(0.02, 0.35))
        rows.append({
            "mention_id": mid,
            "timestamp": iso_ts(ts),
            "platform": random.choice(platforms),
            "sentiment": random.choice(sentiments),
            "reach_estimate": reach,
            "engagement": engagement
        })
    return pd.DataFrame(rows)

def generate_summary(df_users, df_health, df_chat, df_logs):
    suspicious_total = int(df_logs["is_suspicious"].sum())
    weak_pw_users = int((df_users["password_plain"].isin(["123456", "password", "azerty123", "admin123", "motdepasse"])).sum())

    summary_rows = [
        ("Total Users", len(df_users)),
        ("Total Health Records", len(df_health)),
        ("Total Conversations", len(df_chat)),
        ("Total Access Logs", len(df_logs)),
        ("Suspicious Access Attempts", suspicious_total),
        ("Users with Weak Passwords", weak_pw_users),
        ("Data Exposure Duration", "3 months"),
        ("Countries Affected", "France, Belgium")
    ]
    return pd.DataFrame(summary_rows, columns=["Metric", "Value"])

# MAIN
def main():
    df_users = generate_users()
    df_health = generate_health_data(df_users)
    df_chat = generate_chatbot(df_users)
    df_logs = generate_access_logs()
    df_susp_found = derive_suspicious_found(df_logs)
    df_password_resets = generate_password_resets(df_users)
    df_crisis = generate_crisis_notifications(df_susp_found, df_password_resets)
    df_support = generate_support_calls()
    df_social = generate_social_mentions()
    df_summary = generate_summary(df_users, df_health, df_chat, df_logs)

    # Reorder columns exactly as your Excel
    df_users = df_users[[
        "user_id","first_name","last_name","email","password_hash","password_plain",
        "date_of_birth","phone","address","registration_date","country","is_premium","last_login"
    ]]
    df_health = df_health[[
        "user_id","date","timestamp","heart_rate_avg","heart_rate_max","heart_rate_min",
        "steps_count","distance_km","calories_burned","sleep_duration_hours",
        "sleep_quality_score","water_intake_ml","weight_kg"
    ]]
    df_chat = df_chat[[
        "conversation_id","user_id","user_email","timestamp","user_message","bot_response","sentiment","conversation_length"
    ]]
    df_logs = df_logs[[
        "log_id","timestamp","source_ip","user_agent","request_method","endpoint","status_code",
        "response_size_bytes","query_parameters","is_authenticated","is_suspicious"
    ]]
    df_susp_found = df_susp_found[[
        "log_id","timestamp","source_ip","user_agent","request_method","endpoint","status_code",
        "response_size_bytes","query_parameters","is_authenticated","is_suspicious"
    ]]
    df_crisis = df_crisis[[
        "notification_id","timestamp","type","recipient","status","priority","triggered_by"
    ]]
    df_password_resets = df_password_resets[[
        "reset_id","user_id","timestamp","reset_type","status","mfa_enabled_after"
    ]]
    df_support = df_support[[
        "call_id","timestamp","category","duration_minutes","resolved","satisfaction_rating"
    ]]
    df_social = df_social[[
        "mention_id","timestamp","platform","sentiment","reach_estimate","engagement"
    ]]

    # Write Excel with the SAME sheet names & order
    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
        df_users.to_excel(writer, sheet_name="Users", index=False)
        df_health.to_excel(writer, sheet_name="Health Data", index=False)
        df_chat.to_excel(writer, sheet_name="Chatbot", index=False)
        df_logs.to_excel(writer, sheet_name="Access Logs", index=False)
        df_susp_found.to_excel(writer, sheet_name="Suspisious found", index=False)
        df_summary.to_excel(writer, sheet_name="Summary", index=False)
        df_crisis.to_excel(writer, sheet_name="Crisis Notifications", index=False)
        df_password_resets.to_excel(writer, sheet_name="Password Resets", index=False)
        df_support.to_excel(writer, sheet_name="Support Calls", index=False)
        df_social.to_excel(writer, sheet_name="Social Mentions", index=False)

    print(f"✅ Generated: {OUTPUT_XLSX}")
    print("Sheets:", [
        "Users","Health Data","Chatbot","Access Logs","Suspisious found","Summary",
        "Crisis Notifications","Password Resets","Support Calls","Social Mentions"
    ])

if __name__ == "__main__":
    main()

✅ Generated: /tmp/healthyou_db.xlsx
Sheets: ['Users', 'Health Data', 'Chatbot', 'Access Logs', 'Suspisious found', 'Summary', 'Crisis Notifications', 'Password Resets', 'Support Calls', 'Social Mentions']


In [3]:
pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install Faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 36.8 MB/s  0:00:00
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
Note: you may need to restart the kernel to use updated packages.
